# 05: Agrupar datos: GROUP BY y HAVING

La analogía: 

1. imagina que tienes una caja gigante con todos los recibos del año. 
2. GROUP BY es como separar los recibos en montones:
    - uno por ciudad
    - uno por mes
    - uno por vendedor. 
3. Luego cuentas/sumas cada montón por separado. 

El resultado es una fila por grupo con sus métricas.



## GROUP BY: dividir la tabla en grupos


```sql
-- Número de clientes por ciudad
SELECT ciudad, COUNT(*) AS total_clientes,
FROM clientes
GROUP BY ciudad
ORDER BY total_clientes DESC;

-- VEntas totales por estado del pedido
SELECT estado, COUNT(*) AS pedidos, ROUND(SUM(importe), 2) AS total
FROM pedidos
GROUP BY estado;

-- Vendedores por equipo
SELECT equipo, COUNT(*) AS vendedores, ROUND(SUM(ventas), 2) AS total_ventas
FROM vendedores
GROUP BY equipo
ORDER BY total_ventas DESC;
```

**GROUP BY:** una fila por cada valor único de la columna agrupada

Tres ejemplos de GROUP BY con diferentes tablas:

- GROUP BY ciudad + COUNT(*) → cuenta clientes por ciudad. Resultado: tantas filas como ciudades únicas.
- GROUP BY estado + COUNT + SUM → para cada estado de pedido, cuántos hay y cuánto suman.
- GROUP BY equipo + COUNT + SUM → vendedores y ventas totales por equipo.

El patrón es siempre el mismo: GROUP BY columna + función(es) de agregación. 

Cada valor único de la columna agrupada genera exactamente una fila en el resultado.



## La regla de oro de GROUP BY



Cuando usas GROUP BY, TODA columna en el SELECT debe ser: (a) parte del GROUP BY, o (b) una función de agregación. 

- No puedes mezclar columnas individuales con agregados sin agrupar.
- Es un error que todo junior comete al menos una vez:

    -- [OK] CORRECTO: ciudad está en GROUP BY, COUNT es agregación
SELECT ciudad, COUNT(*) AS total
FROM clientes
GROUP BY ciudad;

-- [ERROR] ERROR: nombre NO está en GROUP BY ni es agregación
-- SELECT ciudad, nombre, COUNT(*) FROM clientes GROUP BY ciudad;
-- Error: column "nombre" must appear in GROUP BY clause

-- [OK] Si quieres nombre, agrúpalo también
SELECT ciudad, nombre, COUNT(*) AS total
FROM clientes
GROUP BY ciudad, nombre;

```sql
-- CORRECTO: ciudad está en GROUP BY; COUNT es agregación
SELECT ciudad, COUNT(*) AS total
FROM clientes
GROUP BY ciudad;

-- ERROR: nombrfe no está en GROUP BY ni en agregación
SELECT ciudad, nombre, COUNT(*)
FROM clientes
GROUP BY ciudad; -- Erorr: column "nombre" must appear in GROUP BY clause

-- CORRECTO: si quires nombre, agrúpalo también
SELECT ciudad, nombre, COUNT(*) AS total
FROM clientes
GROUP BY ciudad, nombre;
```

**La regla:** en SELECT solo columnas del GROUP BY o funciones de agregación

La regla de oro de GROUP BY demostrada:

- [OK] SELECT ciudad, COUNT(*) → ciudad está en GROUP BY, COUNT es agregación. Correcto.
- [ERROR] SELECT ciudad, nombre, COUNT(*) GROUP BY ciudad → "nombre" no está en GROUP BY ni es agregación. ¿Qué nombre mostraría si hay 500 clientes en Madrid? El motor no puede elegir uno.
- [OK] GROUP BY ciudad, nombre → ahora cada combinación ciudad+nombre es un grupo. Funciona pero da muchos más grupos.

Esta regla existe porque GROUP BY colapsa múltiples filas en una. 

Si una columna no está agrupada ni agregada, el motor no sabe qué valor de las N filas del grupo devolver.

## Agrupar por múltiples columnas


Puedes agrupar por 2 o más columnas. 

1. Cada combinación única de valores genera un grupo.
2. Es como hacer una tabla cruzada:
    - ventas por ciudad Y por categoría


-- Pedidos por estado y mes (2 columnas en GROUP BY)
SELECT 
    estado,
    DATE_TRUNC('month', fecha) AS mes,
    COUNT(*) AS pedidos,
    ROUND(SUM(importe), 2) AS total
FROM pedidos
WHERE fecha >= '2024-01-01'
GROUP BY estado, DATE_TRUNC('month', fecha)
ORDER BY mes, estado;

```sql
-- Pedidos por estado y mes (2 columnas en GROUP BY)
SELECT 
    estado, 
    DATE_TRUNC('month', fecha) AS mes, 
    COUNT(*) AS pedidos
    ROUND(SUM(importe), 2) AS total
FROM pedidos
WHERE fecha >= '2024-01-01' 
GROUP BY estado, DATE_TRUNC('month', fecha) 
ORDER BY mes, estado;
```

**GROUP BY con dos columnas:** cada combinación única es un grupo

- GROUP BY estado, DATE_TRUNC('month', fecha) → crea un grupo por cada combinación estado+mes. Si hay 3 estados y 12 meses, podrían ser hasta 36 grupos.
- DATE_TRUNC('month', fecha) → trunca la fecha al primer día del mes (2024-03-15 → 2024-03-01). Esto agrupa todos los días de un mes juntos.
- WHERE fecha >= '2024-01-01' → filtra ANTES de agrupar (solo datos de 2024).
- ORDER BY mes, estado → ordena cronológicamente y dentro de cada mes, por estado.

Multiple GROUP BY crea una tabla cruzada:
- el embrión de un pivot table en Excel.

Dos atajos que vas a ver en consultas ajenas y en algún ejercicio. 

1. El primero es ::DATE, que se lee «conviértelo a fecha»: 
    - Según el motor y su versión, DATE_TRUNC('month', fecha) puede devolverte el mes con hora incluida (2024-03-01 00:00:00),
    - DATE_TRUNC('month', fecha)::DATE te deja solo 2024-03-01. 
    - Los dos puntos dobles sirven para cualquier tipo: '42'::INTEGER convierte el texto en número. 
2. El segundo es GROUP BY 1, que agrupa por la primera columna del SELECT sin tener que repetirla; 
    - GROUP BY 1, 2 agrupa por la primera y la segunda. 
    - La consulta de arriba podría terminar en GROUP BY 1, 2, porque estado es la primera columna y el mes la segunda. 
    - Es cómodo, pero si alguien cambia el orden del SELECT, el grupo cambia sin avisar: en consultas que se quedan, mejor escribir los nombres.



## HAVING: filtrar DESPUÉS de agrupar


1. WHERE filtra filas individuales ANTES de agrupar. 
2. Pero ¿qué haces cuando quieres filtrar los GRUPOS? 
    - "Ciudades con más de 1000 clientes",
    -  "categorías con ventas superiores a 100K"... 

No puedes usar WHERE porque la condición depende de la agregación. 

Para eso existe HAVING:

```sql
-- Ciudades con más de 900 clientes
SELECT ciudad, COUNT(*) AS total_clientes
FROM clientes
GROUP BY ciudad
HAVING COUNT(*) > 900
ORDER BY total_clientes DESC;

-- Clientes que han gastado más de 800 en total
SELECT cliente_id, ROUND(SUM(importe), 2) AS gasto_total
FROM pedidos
WHERE estado = 'completado'
GROUP BY cliente_id
HAVING SUM(importe) > 800
ORDER BY gasto_total DESC
LIMIT 10;
```

**HAVING filtra grupos, WHERE filtra filas. Orden: WHERE → GROUP → HAVING**

Dos ejemplos de HAVING:

- HAVING COUNT(*) > 900 → después de agrupar por ciudad, descarta ciudades con menos de 900 clientes. Solo las "grandes" pasan.
- HAVING SUM(importe) > 800 → primero filtra filas (WHERE estado completado), luego agrupa por cliente, y finalmente descarta clientes cuyo gasto total es ≤ 800.

- El flujo es: FROM → WHERE (filas) → GROUP BY → HAVING (grupos) → SELECT → ORDER BY → LIMIT. 
- HAVING trabaja DESPUÉS de GROUP BY, por eso puede usar funciones de agregación.

## WHERE vs HAVING: la diferencia clave


- WHERE filtra FILAS INDIVIDUALES antes de agrupar. 
  - Ejemplo: "solo pedidos completados".
- HAVING filtra GRUPOS después de agrupar. 
  - Ejemplo: "solo ciudades con más de 100 pedidos".
- WHERE no puede usar funciones de agregación 
  - COUNT, SUM, AVG...
- HAVING SÍ puede usar funciones de agregación: 
  - para eso existe.
- Puedes usar AMBOS en la misma query: 
  1. WHERE reduce filas, 
  2. luego HAVING reduce grupos.

```sql
-- WHERE + HAVING combinados:
-- "DE los pedidos completados (WHERE),
-- muestra solo las ciudades con más de 500 pedidos (HAVING)"
SELECT c.ciudad, COUNT(*) AS pedidos, ROUND(SUM(p.importe), 2) AS total
FROm cliente c
JOIN pedidos p ON p.clientes_id = c.id
WHERE p.estado = 'completado'           -- filtra filas ANTES de agrupar
GROUP BY c.ciudad
HAVING COUNT(*) > 500                   -- filtra grupos DESPUËS de agrupar
ORDER BY total DESC;
```

**WHERE y HAVING trabajan juntos:** primero filas, luego grupos

Combinación de ambos filtros:

- WHERE p.estado = 'completado': 
  - ejecutado ANTES de GROUP BY. 
  - Descarta pedidos cancelados/devueltos a nivel de fila individual.
- GROUP BY c.ciudad: 
  - agrupa las filas restantes por ciudad.
- HAVING COUNT(*) > 500: 
  - ejecutado DESPUÉS de GROUP BY. 
  - Descarta ciudades con menos de 500 pedidos completados.

**Piénsalo en dos pasos:**

1. WHERE es el primer filtro (quita lo que no interesa), GROUP BY resume, 
2. HAVING es el segundo filtro (quita grupos pequeños/irrelevantes).

Este ejemplo también incluye un JOIN (clientes + pedidos), lo que demuestra que WHERE/GROUP BY/HAVING funcionan igual con datos de múltiples tablas.



**Truco mnemotécnico:**

- WHERE trabaja con datos "crudos" 
  - (antes del resumen). 
- HAVING trabaja con datos "cocidos" 
  - (después del resumen). 

1. Si tu condición usa COUNT, SUM, AVG
    - va en HAVING. 
2. Si no 
    - va en WHERE.



## Orden de ejecución de SQL (modelo mental)



SQL NO se ejecuta en el orden en que lo escribes. 

El motor procesa las cláusulas en este orden interno, y entenderlo te ahorra muchos errores:

01. **FROM:** elige la tabla (o tablas con JOIN)
02. **WHERE:** filtra filas individuales
03. **GROUP BY:** agrupa las filas que pasaron el filtro
04. **HAVING:** filtra los grupos resultantes
05. **SELECT:** calcula las expresiones y columnas del resultado
06. **ORDER BY:** ordena el resultado final
07. **LIMIT:** recorta el número de filas devueltas

## Contar solo una parte del grupo: COUNT(CASE WHEN ...)


COUNT(*) cuenta todas las filas del grupo. 

A menudo quieres contar solo algunas: 
- cuántos clientes de cada ciudad son VIP, 
- cuántos pedidos de cada mes se cancelaron. 

Para eso se mete un CASE WHEN dentro del COUNT. 

CASE WHEN es el «si…, entonces…» de SQL: 

- CASE WHEN es_vip THEN 1 END se lee «si la fila es VIP, vale 1». 
- Cuando la condición no se cumple y no hay nada más escrito, vale NULL. 
- como COUNT(columna) se salta los NULL , el resultado es justo el número de filas que cumplen la condición. 

CASE da para mucho más, y tiene su propia lección más adelante; para contar, con esta forma te basta:

```sql
SELECT
    ciudad, 
    COUNT(*) AS total,
    COUNT(CASE WHEN es_vip THEN 1 END) AS vips
FROM clientes
GROUP BY ciudad;
```

1. COUNT(*) cuenta todo el grupo; 
2. COUNT(CASE WHEN ...) solo las filas que cumplen la condición

- COUNT(*) AS total: todas las filas de cada ciudad.
- CASE WHEN es_vip THEN 1 END: para cada fila: si es VIP vale 1; si no, NULL.
- COUNT( … ) AS vips: cuenta los valores que no son NULL, o sea, solo los VIP.
- GROUP BY ciudad: todo lo anterior se calcula una vez por ciudad.

Si quieres el porcentaje, divide uno entre otro: 

  - 100.0 * COUNT(CASE WHEN es_vip THEN 1 END) / COUNT(*). 
  - El 100.0 con decimales asegura que la división dé decimales: 
  - En muchas bases de datos, dividir dos enteros da otro entero (5 / 40 = 0).



## FILTER: la alternativa elegante a COUNT(CASE WHEN...)

El patrón COUNT(CASE WHEN condicion THEN 1 END) funciona pero es verboso. 

- En DuckDB y PostgreSQL (9.4+) existe una forma más limpia de contar subconjuntos: 
    - la cláusula FILTER. 

- Hace exactamente lo mismo pero se lee mucho mejor:



```sql
-- Forma clásica (verbosa pero universal)
SELECT
    ciudad, 
    COUNT(*) AS total,
    COUNT(CASE WHEN es_vip THEN 1 END) AS vips
FROM clientes
GROUP BY ciudad;

-- Forma moderna con FILTER (DuckDB, PostgreSQL 9.4+)
SELECT
    ciudad,
    COUNT(*) AS total,
    COUNT(*) FILTER (WHERE es_vip) AS vips
FROM clientes
GROUP BY ciudad;

```

**FILTER(WHERE...) es más legible:** la condición va separada de la función de agregación


Ambas queries devuelven exactamente lo mismo. La diferencia es legibilidad:

- COUNT(CASE WHEN es_vip THEN 1 END): 
    - la condición está "escondida" dentro del CASE.
- COUNT(*) FILTER (WHERE es_vip): 
    - la condición va claramente separada. 
    - Se lee como "cuenta todas, FILTRADAS por las que son VIP".

FILTER funciona con cualquier función de agregación: 
  - SUM(...) FILTER (WHERE ...), AVG(...) FILTER (WHERE ...), etc.

**Consejo de senior:** en entrevistas técnicas, si usas FILTER demuestras que conoces SQL moderno. 

En producción, usa lo que tu equipo use
  - la consistencia importa más que la elegancia.



**Consejo de senior:** 

1. en mi equipo actual todos usamos FILTER porque el código se revisa más rápido 
   - la condición está a la vista en vez de enterrada en un CASE WHEN. 
2. Pero si trabajas con MySQL, no lo tiene (solo DuckDB, PostgreSQL, SQLite 3.30+). 

Aprende ambas formas: COUNT(CASE WHEN...) es universal, FILTER es el futuro.

